In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
BASE = Path("../data/raw/Avonet")

# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings from messy raw data
import warnings
warnings.filterwarnings("ignore")

In [ ]:
avonet = pd.read_csv(BASE / "avonet_uncleaned_final_dataset.csv")
avonet_crosswalk = pd.read_csv(BASE / "avonet_crosswalk.csv")

In [ ]:
avonet = avonet.merge(
     avonet_crosswalk[["avibase_id", "order_birdlife"]],
    on="avibase_id",
    how="left"
)

In [ ]:
columns = avonet.columns.tolist()

In [ ]:
quantitative_feature = ['beak_culmen_avg', 'beak_culmen_avg_m', 'beak_culmen_avg_f', 'beak_nares_avg',
                       'beak_nares_avg_m', 'beak_nares_avg_f', 'beak_width_avg', 'beak_width_avg_m', 'beak_width_avg_f',
                       'beak_depth_avg', 'beak_depth_avg_m', 'beak_depth_avg_f', 'tarsus_avg', 'tarsus_avg_m', 'tarsus_avg_f',
                       'wing_len_avg', 'wing_len_avg_m', 'wing_len_avg_f', 'kipps_avg', 'kipps_avg_m', 'kipps_avg_f', 'secondary_avg',
                       'secondary_avg_m', 'secondary_avg_f', 'hwi_avg', 'hwi_avg_m', 'hwi_avg_f', 'tail_avg', 'tail_avg_m', 'tail_avg_f',
                         'mass_avg', 
                        'lat_min', 'lat_max', 'lat_centroid', 'lon_centroid', 'range_size']

categorical_feature = [  'habitat', 'habitat_density', 'migration', 'trophic_level', 
                       'trophic_niche', 'lifestyle']

# COVERAGE

### Feature-level coverage

In [ ]:
cols =  quantitative_feature + categorical_feature

In [ ]:
def feature_coverage(df, features):
    records = []

    total = len(df)

    for col in features:
        non_null = df[col].notna().sum()
        coverage = non_null / total * 100

        records.append({
            "feature": col,
            "non_null_count": non_null,
            "total": total,
            "coverage_%": round(coverage, 2)
        })

    return pd.DataFrame(records).sort_values(by="coverage_%")

In [ ]:
feature_coverage(avonet,cols)

### Taxonomy-level coverage

In [ ]:
def taxonomy_coverage(df, features, group_col="order_birdlife"):
    records = []

    grouped = df.groupby(group_col)

    for group, gdf in grouped:
        total = len(gdf)

        for col in features:
            non_null = gdf[col].notna().sum()
            coverage = non_null / total * 100

            records.append({
                "order": group,
                "feature": col,
                "non_null": non_null,
                "total": total,
                "coverage_%": round(coverage, 2)
            })

    return pd.DataFrame(records).sort_values(by="coverage_%")

In [ ]:
temp = taxonomy_coverage(avonet,cols)

In [ ]:
agg = (
        temp
        .groupby("order")
        .agg({
            "non_null": "sum",
            "total": "sum"
        })
        .reset_index()
    )

# --- compute coverage ---
agg["coverage_%"] = (agg["non_null"] / agg["total"]) * 100

    # --- sort ---
agg = agg.sort_values(by="coverage_%")

    # --- filter low coverage ---
low_coverage = agg[agg["coverage_%"] < 80]

In [ ]:
agg

### conclusion 
- all order have covarage greate than 80%

## Infered

In [ ]:
def inferred_global(df, col="inference"):
    counts = df[col].value_counts(dropna=False)

    total = len(df)

    records = []
    for k, v in counts.items():
        records.append({
            "type": k,
            "count": v,
            "percentage": round(v / total * 100, 2)
        })

    return pd.DataFrame(records)

In [ ]:
def inferred_by_order(df, group_col="order_birdlife", col="inference"):
    records = []

    grouped = df.groupby(group_col)

    for group, gdf in grouped:
        total = len(gdf)
        counts = gdf[col].value_counts(dropna=False)

        for k, v in counts.items():
            records.append({
                "order": group,
                "type": k,
                "count": v,
                "percentage": round(v / total * 100, 2)
            })

    return pd.DataFrame(records)

In [ ]:
inferred_global(avonet)

In [ ]:
temp  = inferred_by_order(avonet)

In [ ]:
temp = temp[temp["type"] == "NO"].sort_values(by="percentage")

In [ ]:
temp

### conclusion 
- all order have covarage greate than 82%

## Categorical Imbalance

In [ ]:
def categorical_distribution(df, features):
    records = []

    for col in features:
        counts = df[col].value_counts(normalize=True, dropna=True)

        for category, pct in counts.items():
            records.append({
                "feature": col,
                "category": category,
                "percentage": round(pct * 100, 2)
            })

    return pd.DataFrame(records)

In [ ]:
from scipy.stats import entropy

def categorical_imbalance(df, features):
    records = []

    for col in features:
        counts = df[col].value_counts(normalize=True, dropna=True)

        if len(counts) <= 1:
            imbalance = 100.0
        else:
            ent = entropy(counts)
            max_ent = np.log(len(counts))
            imbalance = (1 - ent / max_ent) * 100

        records.append({
            "feature": col,
            "num_categories": len(counts),
            "imbalance_%": round(imbalance, 2)
        })

    return pd.DataFrame(records).sort_values(by="imbalance_%", ascending=False)

In [ ]:
temp = categorical_distribution(avonet,categorical_feature)

In [ ]:
temp

In [ ]:
categorical_imbalance(avonet,categorical_feature)

In [ ]:
def feature_level_imbalance(cat_df):

    records = []

    grouped = cat_df.groupby("feature")

    for feature, gdf in grouped:
        # dominant category
        max_row = gdf.loc[gdf["percentage"].idxmax()]
        dominant_cat = max_row["category"]
        dominant_pct = max_row["percentage"]

      
        imbalance = dominant_pct

        if imbalance > 60:
            cls = "Highly Imbalanced"
        elif imbalance > 40:
            cls = "Imbalanced"
        elif imbalance > 25:
            cls = "Moderate"
        else:
            cls = "Balanced"

        records.append({
            "feature": feature,
            "dominant_category": dominant_cat,
            "dominant_%": round(dominant_pct, 2),
            "imbalance_%": round(imbalance, 2),
            "class": cls
        })

    return pd.DataFrame(records).sort_values(by="imbalance_%", ascending=False)

In [ ]:
feature_level_imbalance(temp)

# Conclusion

#### 1. Feature-Level Coverage
-  All features show high coverage, with most exceeding 80% completeness across the dataset. Core morphological traits are particularly well represented, indicating strong overall data availability.
#### 2. Taxonomy-Level Coverage
-  Coverage across all taxonomic orders is consistently above 80%, suggesting no major gaps in data availability at the group level. This indicates the dataset is broadly complete across different bird orders.
#### 3. Categorical Imbalance
-  Most categorical traits exhibit significant imbalance, with certain categories dominating (e.g., migration and habitat). This suggests ecological bias in the dataset, potentially underrepresenting less common behaviors and environments